In [2]:
import pandas as pd
df = pd.read_csv("loan_data.csv")
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [4]:
df["loan_status"].value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [6]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

In [7]:
print(X.shape)
print(y.shape)

(45000, 13)
(45000,)


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

test_size=0.2 → %20 test
stratify=y → 0/1 oranını train ve testte koru
random_state=42 → her çalıştırmada aynı bölünmeyi elde et

In [10]:
print(X_train.shape)
print(X_test.shape)

(36000, 13)
(9000, 13)


In [11]:
print(y_train.value_counts())
print(y_test.value_counts())

loan_status
0    28000
1     8000
Name: count, dtype: int64
loan_status
0    7000
1    2000
Name: count, dtype: int64


stratify=y kullanmazsak train ve test setlerinde 0 ve 1 sınıflarının dağılımı farklılaşabilir. Bu da özellikle dengesiz sınıflarda test setinin temsil gücünü etkileyebilir.

In [12]:
from sklearn.metrics import accuracy_score

In [13]:
len(y_train)

36000

In [15]:
baseline_predictions = [0] * len(y_train)
print(len(baseline_predictions))

36000


In [16]:
baseline_accuracy = accuracy_score(y_train, baseline_predictions)
print(baseline_accuracy)

0.7777777777777778


In [17]:
from sklearn.preprocessing import OneHotEncoder

In [19]:
categorical_features = [
    "person_gender",
    "person_education",
    "person_home_ownership",
    "loan_intent",
    "previous_loan_defaults_on_file"
]

numerical_features = [
    "person_age",
    "person_income",
    "person_emp_exp",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "credit_score"
]

In [20]:
from sklearn.compose import ColumnTransformer

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

"cat" → bizim verdiğimiz isim
OneHotEncoder(...) → kategorik sütunları dönüştürüyor
"num" → sayısal sütunlar için isim
"passthrough" → sayısal sütunları olduğu gibi bırak
categorical_features → az önce oluşturduğun liste
numerical_features → az önce oluşturduğun liste

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [24]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [26]:
model.fit(X_train, y_train)

c:\Users\esmah\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['person_gender',
                                                   'person_education',
                                                   'person_home_ownership',
                                                   'loan_intent',
                                                   'previous_loan_defaults_on_file']),
                                                 ('num', 'passthrough',
                                                  ['person_age',
                                                   'person_income',
                                                   'person_emp_exp',
                                                   'loan_amnt', 'loan_int_rate',
                                                   'loan_percent_income',
                                                   'cb_person_cred_hist_length',
                                                   'credit_score'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [27]:
y_pred = model.predict(X_test)

In [28]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8883333333333333


In [29]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[6526  474]
 [ 531 1469]]


In [30]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)

Precision: 0.7560473494595985
Recall: 0.7345


In [31]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
print("F1:", f1)

F1: 0.7451179305097642


“Accuracy yüksek” ≠ “model her açıdan iyi.”

Özellikle sınıf dengesizliğinde confusion matrix + precision + recall + F1 birlikte okunmalı.